# Runtime comparison

This notebook compares the PyTheus and surrogate inverse-design runtimes for GHZ, W, and linear-cluster (LC) targets at $n=4,6,8$. Bars show the mean of 50 samples and asymmetric error bars span the observed minimum and maximum.

The first surrogate timing is excluded as a compilation/warm-up measurement.

## Setup

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np


def find_repository_root():
    """Support launching Jupyter from the repository root or plotting_code/."""
    current = Path.cwd().resolve()
    if (current / "ml_model.py").is_file() and (current / "plotting_code").is_dir():
        return current
    if current.name == "plotting_code" and (current.parent / "ml_model.py").is_file():
        return current.parent
    raise RuntimeError(
        "Launch this notebook from the PolySurrogate repository root "
        "or from its plotting_code directory."
    )


REPO_ROOT = find_repository_root()
PLOTTING_DIR = REPO_ROOT / "plotting_code"
RUNTIME_DATA_DIR = PLOTTING_DIR / "Data" / "runtime_data"
PYTHEUS_DIR = RUNTIME_DATA_DIR / "pytheus_data_time"
SURROGATE_DIR = RUNTIME_DATA_DIR / "surrogate_optimisation"
OUTPUT_DIR = PLOTTING_DIR / "Results" / "runtime_bar"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

STATES = ("GHZ", "W", "LC")
NODE_COUNTS = (4, 6, 8)
MAX_SAMPLES = 50
SAVE_FORMATS = ("png", "pdf", "svg")
SAVE_DPI = 600

FIGSIZE = (11.5, 10)
BAR_WIDTH = 0.36
BAR_EDGE_WIDTH = 1.5
ERROR_BAR_CAPSIZE = 6
ERROR_BAR_LINEWIDTH = 1.7
TICK_FONTSIZE = 23
AXIS_LABEL_FONTSIZE = 25
LEGEND_FRAME_ALPHA = 0.92
LEGEND_EDGE_COLOR = "0.75"

plt.rcParams.update({
    "font.size": 28,
    "axes.titlesize": 30,
    "axes.labelsize": 34,
    "xtick.labelsize": 28,
    "ytick.labelsize": 28,
    "legend.fontsize": 28,
    "lines.linewidth": 2.4,
    "axes.linewidth": 1.3,
    "xtick.major.width": 1.2,
    "ytick.major.width": 1.2,
    "xtick.major.size": 5.0,
    "ytick.major.size": 5.0,
    "figure.dpi": 150,
    "savefig.dpi": SAVE_DPI,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

PYTHEUS_FILES = {
    state: {n: PYTHEUS_DIR / state / f"{n}n_time_info.json" for n in NODE_COUNTS}
    for state in STATES
}
SURROGATE_FILES = {
    state: {n: SURROGATE_DIR / state / f"{n}n.json" for n in NODE_COUNTS}
    for state in STATES
}

all_input_files = [
    path
    for files in (PYTHEUS_FILES, SURROGATE_FILES)
    for state_files in files.values()
    for path in state_files.values()
]
missing_files = [path for path in all_input_files if not path.is_file()]
if missing_files:
    formatted = "\n".join(f"  - {path}" for path in missing_files)
    raise FileNotFoundError(f"Missing runtime data files:\n{formatted}")

print("Runtime data :", RUNTIME_DATA_DIR)
print("Figure output:", OUTPUT_DIR)
print(f"Validated {len(all_input_files)} input files.")

## Load timing data and calculate statistics

In [ ]:
def load_json(path):
    with path.open() as file:
        return json.load(file)


def clean_times(values, max_samples):
    cleaned = []
    for value in values:
        try:
            value = float(value)
        except (TypeError, ValueError):
            continue
        if np.isfinite(value):
            cleaned.append(value)
    return np.asarray(cleaned[:max_samples], dtype=float)


def extract_times(path, source):
    data = load_json(path)
    if source == "PyTheus":
        key = "Time_pre_opt"
        values = data.get(key, [])
    else:
        key = "time_per_sample"
        values = data.get(key, [])[1 : MAX_SAMPLES + 1]

    if key not in data:
        raise KeyError(f"{key!r} not found in {path}")
    times = clean_times(values, MAX_SAMPLES)
    if times.size == 0:
        raise ValueError(f"No finite timing values found in {path}")
    return times


def summarize(times):
    mean = times.mean()
    minimum = times.min()
    maximum = times.max()
    return {
        "mean": mean,
        "lower": mean - minimum,
        "upper": maximum - mean,
        "min": minimum,
        "max": maximum,
        "std": times.std(),
        "num_samples": times.size,
    }


statistics = {state: {"PyTheus": {}, "Surrogate": {}} for state in STATES}
for state in STATES:
    for node_count in NODE_COUNTS:
        pytheus_times = extract_times(PYTHEUS_FILES[state][node_count], "PyTheus")
        surrogate_times = extract_times(SURROGATE_FILES[state][node_count], "Surrogate")
        statistics[state]["PyTheus"][node_count] = summarize(pytheus_times)
        statistics[state]["Surrogate"][node_count] = summarize(surrogate_times)

        pytheus = statistics[state]["PyTheus"][node_count]
        surrogate = statistics[state]["Surrogate"][node_count]
        print(
            f"{state}, n={node_count}: "
            f"PyTheus={pytheus['mean']:.6f}s, "
            f"Surrogate={surrogate['mean']:.6f}s "
            f"({pytheus['num_samples']}/{surrogate['num_samples']} samples)"
        )

## Save the numerical summary

In [ ]:
summary_path = OUTPUT_DIR / "runtime_barplot_summary_minmax.txt"
with summary_path.open("w") as file:
    for state in STATES:
        file.write(f"\n{state}\n")
        file.write("=" * 80 + "\n")
        for node_count in NODE_COUNTS:
            pytheus = statistics[state]["PyTheus"][node_count]
            surrogate = statistics[state]["Surrogate"][node_count]
            file.write(
                f"n={node_count} | "
                f"PyTheus samples={pytheus['num_samples']}, "
                f"PyTheus mean={pytheus['mean']:.10f}, "
                f"PyTheus std={pytheus['std']:.10f}, "
                f"PyTheus min={pytheus['min']:.10f}, "
                f"PyTheus max={pytheus['max']:.10f}, "
                f"Surrogate samples={surrogate['num_samples']}, "
                f"Surrogate mean={surrogate['mean']:.10f}, "
                f"Surrogate std={surrogate['std']:.10f}, "
                f"Surrogate min={surrogate['min']:.10f}, "
                f"Surrogate max={surrogate['max']:.10f}\n"
            )

print("Saved:", summary_path)

## Generate logarithmic and linear runtime plots

In [ ]:
def prepare_plot_data():
    labels = []
    pytheus_means, surrogate_means = [], []
    pytheus_lower, pytheus_upper = [], []
    surrogate_lower, surrogate_upper = [], []

    for state in STATES:
        for node_count in NODE_COUNTS:
            labels.append(f"{state}-{node_count}")
            pytheus = statistics[state]["PyTheus"][node_count]
            surrogate = statistics[state]["Surrogate"][node_count]
            pytheus_means.append(pytheus["mean"])
            surrogate_means.append(surrogate["mean"])
            pytheus_lower.append(pytheus["lower"])
            pytheus_upper.append(pytheus["upper"])
            surrogate_lower.append(surrogate["lower"])
            surrogate_upper.append(surrogate["upper"])

    return (
        labels, np.asarray(pytheus_means), np.asarray(surrogate_means),
        np.asarray([pytheus_lower, pytheus_upper]),
        np.asarray([surrogate_lower, surrogate_upper]),
    )


def style_axis(ax):
    ax.grid(False)
    ax.tick_params(
        axis="both", labelsize=TICK_FONTSIZE, width=1.2,
        length=5.0, direction="out",
    )
    for spine in ax.spines.values():
        spine.set_linewidth(1.3)


def add_legend_headroom(ax, log_scale):
    bottom, top = ax.get_ylim()
    top = top * 10**0.6 if log_scale else top + 0.22 * (top - bottom)
    ax.set_ylim(bottom, top)
    ax.legend(
        loc="upper center", ncol=2, frameon=True,
        framealpha=LEGEND_FRAME_ALPHA, edgecolor=LEGEND_EDGE_COLOR,
        handlelength=1.6, borderpad=0.4, labelspacing=0.3, columnspacing=1.0,
    )


def save_figure(fig, filename):
    fig.tight_layout(pad=1.5)
    for extension in SAVE_FORMATS:
        path = OUTPUT_DIR / f"{filename}.{extension}"
        dpi = SAVE_DPI if extension == "png" else None
        fig.savefig(path, dpi=dpi, bbox_inches="tight")
        print(f"Saved: {path}")
    plt.show()
    plt.close(fig)


def plot_runtime(log_scale):
    labels, pytheus_means, surrogate_means, pytheus_error, surrogate_error = (
        prepare_plot_data()
    )
    x = np.arange(len(labels))
    error_style = {
        "elinewidth": ERROR_BAR_LINEWIDTH,
        "capthick": ERROR_BAR_LINEWIDTH,
    }

    fig, ax = plt.subplots(figsize=FIGSIZE)
    ax.bar(
        x - BAR_WIDTH / 2, pytheus_means, width=BAR_WIDTH, yerr=pytheus_error,
        capsize=ERROR_BAR_CAPSIZE, error_kw=error_style, edgecolor="black",
        linewidth=BAR_EDGE_WIDTH, label="PyTheus",
    )
    ax.bar(
        x + BAR_WIDTH / 2, surrogate_means, width=BAR_WIDTH, yerr=surrogate_error,
        capsize=ERROR_BAR_CAPSIZE, error_kw=error_style, edgecolor="black",
        linewidth=BAR_EDGE_WIDTH, label="Surrogate",
    )

    ax.set_ylabel("Average time (s)", fontsize=AXIS_LABEL_FONTSIZE)
    ax.set_xticks(x)
    ax.set_xticklabels(
        labels, rotation=35, ha="right", fontsize=AXIS_LABEL_FONTSIZE,
    )
    if log_scale:
        ax.set_yscale("log")

    style_axis(ax)
    ax.set_box_aspect(0.87)
    add_legend_headroom(ax, log_scale)

    scale_name = "logscale" if log_scale else "linear"
    save_figure(
        fig, f"grouped_runtime_barplot_pytheus_vs_surrogate_minmax_{scale_name}",
    )


plot_runtime(log_scale=True)
plot_runtime(log_scale=False)